In [ ]:
#!/usr/bin/env python
# coding: utf-8

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
import torch; print(torch.__version__); print(torch.cuda.is_available())

In [ ]:
# -*- coding: utf-8 -*-
"""
Fixed preprocessing pipeline for SFCN brain-age model.

Folder structure expected:

T1/
├── OAS30001_MR_d0129/
│   ├── anat2/
│   │   ├── sub-OAS30001_ses-d0129_run-1_T1w.json
│   │   └── sub-OAS30001_ses-d0129_run-1_T1w.nii.gz
│   └── anat3/
│       └── ...
│
├── OAS30001_MR_d0757/
│   └── anat2/
│       └── ...
│
└── ...

The batch preprocessing searches recursively through all
subdirectories inside T1.
"""

import os
import glob
import ants
import time
import antspynet
import numpy as np
import torchio as tio
import nibabel as nib


# =========================================================
# MNI152 brain template cache
# =========================================================

_MNI_BRAIN_CACHE = "mni152_brain_1mm.nii.gz"


def get_mni_brain_template():
    """
    Returns a skull-stripped 1mm MNI152 template.

    The template is generated once and cached to disk so that
    brain extraction does not need to be repeated for every scan.
    """

    if os.path.exists(_MNI_BRAIN_CACHE):
        return ants.image_read(_MNI_BRAIN_CACHE)

    print("Creating skull-stripped MNI152 template...")

    mni_head = ants.image_read(
        antspynet.get_antsxnet_data("mni152")
    )

    # brain_extraction() returns a probability image.
    mni_mask_prob = antspynet.brain_extraction(
        mni_head,
        modality="t1"
    )

    # Convert probability mask to binary mask.
    mni_mask_bin = ants.threshold_image(
        mni_mask_prob,
        0.5,
        1.0,
        1,
        0
    )

    mni_brain = ants.mask_image(
        mni_head,
        mni_mask_bin
    )

    ants.image_write(
        mni_brain,
        _MNI_BRAIN_CACHE
    )

    print(
        f"Saved MNI brain template to: "
        f"{_MNI_BRAIN_CACHE}"
    )

    return mni_brain


# =========================================================
# Single MRI preprocessing
# =========================================================

def run_preprocessing(
    input_nii_path: str,
    output_tensor_path: str
):
    """
    Executes the 6-stage preprocessing pipeline:

    1. Reorientation & FOV Cropping
    2. N4 Bias Field Correction
    3. Skull-Stripping
    4. Affine Registration to MNI152
    5. Voxel Resampling & Tensor Formatting
       (1mm isotropic, 160x192x160)
    6. Brain-masked Z-Score Normalization
    """

    print(
        f"\nProcessing: "
        f"{os.path.basename(input_nii_path)}"
    )

    # -------------------------------------------------------------
    # Step 1: Reorientation & FOV Cropping
    # -------------------------------------------------------------

    img = ants.image_read(
        input_nii_path
    )

    print(
        f"  Input intensity range: "
        f"[{img.min():.2f}, {img.max():.2f}]"
    )

    img = ants.reorient_image2(
        img,
        orientation="RAS"
    )

    img_cropped = ants.crop_image(
        img
    )


    # -------------------------------------------------------------
    # Step 2: B1/N4 Bias Field Correction
    # -------------------------------------------------------------

    img_n4 = ants.n4_bias_field_correction(
        img_cropped,
        shrink_factor=2
    )


    # -------------------------------------------------------------
    # Step 3: Brain Extraction / Skull Stripping
    # -------------------------------------------------------------

    mask_prob = antspynet.brain_extraction(
        img_n4,
        modality="t1"
    )

    # Convert probability mask to binary mask.
    mask_bin = ants.threshold_image(
        mask_prob,
        0.5,
        1.0,
        1,
        0
    )

    brain = ants.mask_image(
        img_n4,
        mask_bin
    )

    voxel_count = int(
        mask_bin.sum()
    )

    print(
        f"  Brain mask voxel count: "
        f"{voxel_count}"
    )

    if voxel_count < 100_000:

        raise RuntimeError(
            f"Brain extraction found almost no brain tissue "
            f"({voxel_count} voxels) "
            f"in {os.path.basename(input_nii_path)}. "
            f"This may indicate that the input is not a raw "
            f"T1w MRI scan."
        )


    # -------------------------------------------------------------
    # Step 4: Spatial Registration to 1mm MNI152 Template
    # -------------------------------------------------------------

    mni_brain = get_mni_brain_template()

    print(
        "  Registering brain to MNI152..."
    )

    reg = ants.registration(
        fixed=mni_brain,
        moving=brain,
        type_of_transform="Affine"
    )

    warped_brain = reg[
        "warpedmovout"
    ]


    # -------------------------------------------------------------
    # Step 5: Crop/Pad to exact SFCN matrix
    # -------------------------------------------------------------

    temp_nii = "_temp_reg.nii.gz"

    ants.image_write(
        warped_brain,
        temp_nii
    )

    subject = tio.Subject(
        mri=tio.ScalarImage(
            temp_nii
        )
    )

    crop_pad = tio.CropOrPad(
        (160, 192, 160)
    )

    processed_subject = crop_pad(
        subject
    )

    data = (
        processed_subject
        .mri
        .data
        .squeeze()
        .numpy()
    )

    final_affine = (
        processed_subject
        .mri
        .affine
    )

    # Remove temporary registered image.
    if os.path.exists(temp_nii):
        os.remove(temp_nii)


    # -------------------------------------------------------------
    # Step 6: Brain-Masked Z-Score Normalization
    # -------------------------------------------------------------

    brain_mask = data > 0

    standardized_brain = np.zeros_like(
        data,
        dtype=np.float32
    )

    if np.any(brain_mask):

        mu = np.mean(
            data[brain_mask]
        )

        sigma = np.std(
            data[brain_mask]
        )

        standardized_brain[brain_mask] = (
            data[brain_mask] - mu
        ) / (sigma + 1e-8)


    # -------------------------------------------------------------
    # Save NPZ
    # -------------------------------------------------------------

    output_directory = os.path.dirname(
        output_tensor_path
    )

    if output_directory:
        os.makedirs(
            output_directory,
            exist_ok=True
        )

    np.savez_compressed(
        output_tensor_path,
        data=standardized_brain
    )


    # -------------------------------------------------------------
    # Save NIfTI for Slicer
    # -------------------------------------------------------------

    slicer_nii_path = (
        output_tensor_path
        .replace(
            ".npz",
            ".nii.gz"
        )
    )

    nii_img = nib.Nifti1Image(
        standardized_brain,
        final_affine
    )

    nib.save(
        nii_img,
        slicer_nii_path
    )


    print(
        f"  Saved training tensor: "
        f"{output_tensor_path}"
    )

    print(
        f"  Saved Slicer volume: "
        f"{slicer_nii_path}"
    )


# =========================================================
# FIND ALL MRI FILES RECURSIVELY
# =========================================================

def find_t1_files(raw_dir: str):
    """
    Recursively searches for all NIfTI MRI files inside raw_dir.

    Example input structure:

    T1/
        OAS30001_MR_d0129/
            anat2/
                sub-OAS30001_ses-d0129_run-1_T1w.nii.gz

    The complete file paths are returned.
    """

    nii_files = []

    # Find .nii files
    nii_files.extend(
        glob.glob(
            os.path.join(
                raw_dir,
                "**",
                "*.nii"
            ),
            recursive=True
        )
    )

    # Find .nii.gz files
    nii_files.extend(
        glob.glob(
            os.path.join(
                raw_dir,
                "**",
                "*.nii.gz"
            ),
            recursive=True
        )
    )

    # Remove unwanted files
    nii_files = [
        f
        for f in nii_files
        if not f.endswith(".json")
        and "_preproc" not in os.path.basename(f)
    ]

    return sorted(nii_files)


# =========================================================
# BATCH PREPROCESSING
# =========================================================

def batch_preprocess_t1(
    raw_dir: str,
    output_base_dir: str
):
    """
    Preprocesses every T1w MRI found anywhere inside raw_dir.

    The original folder structure is preserved.

    Example:

    INPUT:

    T1/
    └── OAS30001_MR_d0129/
        └── anat2/
            └── sub-OAS30001_ses-d0129_run-1_T1w.nii.gz


    OUTPUT:

    T1 preprocessed/
    └── OAS30001_MR_d0129/
        └── anat2/
            ├── sub-OAS30001_ses-d0129_run-1_T1w_preproc.npz
            └── sub-OAS30001_ses-d0129_run-1_T1w_preproc.nii.gz
    """

    # ---------------------------------------------------------
    # Make sure output directory exists
    # ---------------------------------------------------------

    os.makedirs(
        output_base_dir,
        exist_ok=True
    )


    # ---------------------------------------------------------
    # Find all MRI files recursively
    # ---------------------------------------------------------

    raw_files = find_t1_files(
        raw_dir
    )


    print(
        "\n=============================================="
    )

    print(
        "T1 BATCH PREPROCESSING"
    )

    print(
        "=============================================="
    )

    print(
        f"Input directory:"
    )

    print(
        f"  {os.path.abspath(raw_dir)}"
    )

    print(
        f"\nOutput directory:"
    )

    print(
        f"  {os.path.abspath(output_base_dir)}"
    )

    print(
        f"\nFound {len(raw_files)} MRI file(s)"
    )

    print(
        "==============================================\n"
    )


    # ---------------------------------------------------------
    # No files found
    # ---------------------------------------------------------

    if len(raw_files) == 0:

        print(
            "ERROR: No NIfTI MRI files were found."
        )

        print(
            f"Searched recursively inside:"
        )

        print(
            os.path.abspath(raw_dir)
        )

        return []


    # ---------------------------------------------------------
    # Process every MRI
    # ---------------------------------------------------------

    failures = []

    for index, input_file in enumerate(
        raw_files,
        start=1
    ):

        filename = os.path.basename(
            input_file
        )


        # -----------------------------------------------------
        # Calculate the path relative to T1
        # -----------------------------------------------------

        relative_path = os.path.relpath(
            input_file,
            raw_dir
        )


        # -----------------------------------------------------
        # Remove MRI extension
        # -----------------------------------------------------

        if relative_path.endswith(
            ".nii.gz"
        ):

            relative_without_extension = (
                relative_path[:-7]
            )

        elif relative_path.endswith(
            ".nii"
        ):

            relative_without_extension = (
                relative_path[:-4]
            )

        else:

            print(
                f"Skipping unsupported file: "
                f"{input_file}"
            )

            continue


        # -----------------------------------------------------
        # Construct output path
        #
        # This preserves the original folder structure.
        # -----------------------------------------------------

        output_relative_path = (
            relative_without_extension
            + "_preproc.npz"
        )


        output_file = os.path.join(
            output_base_dir,
            output_relative_path
        )


        # -----------------------------------------------------
        # Create corresponding output folder
        # -----------------------------------------------------

        output_folder = os.path.dirname(
            output_file
        )

        os.makedirs(
            output_folder,
            exist_ok=True
        )


        # -----------------------------------------------------
        # Skip already processed files
        # -----------------------------------------------------

        if os.path.exists(
            output_file
        ):

            print(
                f"\n[{index}/{len(raw_files)}] "
                f"SKIPPING"
            )

            print(
                f"  Already exists:"
            )

            print(
                f"  {output_file}"
            )

            continue


        # -----------------------------------------------------
        # Display current file
        # -----------------------------------------------------

        print(
            "\n=============================================="
        )

        print(
            f"[{index}/{len(raw_files)}]"
        )

        print(
            "Input:"
        )

        print(
            f"  {input_file}"
        )

        print(
            "Output:"
        )

        print(
            f"  {output_file}"
        )

        print(
            "=============================================="
        )


        # -----------------------------------------------------
        # Run preprocessing
        # -----------------------------------------------------

        try:

            # Start timer for this MRI
            start_time = time.perf_counter()

            run_preprocessing(
                input_file,
                output_file
            )

            # Calculate elapsed time
            elapsed_time = time.perf_counter() - start_time

            print(
                f"\n  Preprocessing time: "
                f"{elapsed_time:.2f} seconds"
            )

            print(
                f"  Preprocessing time: "
                f"{elapsed_time / 60:.2f} minutes"
            )

        except Exception as e:

            print(
                "\nFAILED"
            )

            print(
                f"File:"
            )

            print(
                f"  {input_file}"
            )

            print(
                f"Error:"
            )

            print(
                f"  {e}"
            )

            failures.append(
                input_file
            )


    # ---------------------------------------------------------
    # Final report
    # ---------------------------------------------------------

    print(
        "\n\n=============================================="
    )

    print(
        "BATCH PREPROCESSING COMPLETE"
    )

    print(
        "=============================================="
    )

    print(
        f"Total MRI files : {len(raw_files)}"
    )

    print(
        f"Successful      : "
        f"{len(raw_files) - len(failures)}"
    )

    print(
        f"Failed          : {len(failures)}"
    )

    print(
        f"Output location : "
        f"{os.path.abspath(output_base_dir)}"
    )

    print(
        "=============================================="
    )


    # ---------------------------------------------------------
    # Display failures
    # ---------------------------------------------------------

    if failures:

        print(
            "\nFailed files:"
        )

        for f in failures:

            print(
                f"  {f}"
            )

    else:

        print(
            "\nNo preprocessing failures."
        )


    return failures


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    print(
        "Starting batch preprocessing on T1 folder..."
    )

    batch_preprocess_t1(
        raw_dir="/mnt/d/brain age prep and meth/T1",
        output_base_dir="/mnt/d/brain age prep and meth/T1 preprocessed"
    )

## CELL 4 — Run batch preprocessing
This cell executes the existing 6-step preprocessing pipeline over the full `T1` directory.
It skips files that already have a `_preproc.npz` output, so it is safe to rerun after interruption.


In [ ]:
# =========================================================
# CELL 4 — VERIFY COMPLETED HPC PREPROCESSING
# =========================================================
# IMPORTANT:
# The 6-step preprocessing was already completed on the HPC lab
# for the full dataset. DO NOT rerun preprocessing here.
#
# This cell only verifies that the expected preprocessed NPZ files
# are available to the training pipeline.

import os
import glob
import numpy as np

# FIXED: Linux path for WSL/HPC compatibility
PREPROC_DIR = "/mnt/d/brain age prep and meth/T1 preprocessed"

preproc_files = sorted(
    glob.glob(
        os.path.join(PREPROC_DIR, "**", "*_preproc.npz"),
        recursive=True
    )
)

print("=" * 70)
print("CELL 4: HPC PREPROCESSING VERIFICATION")
print("=" * 70)
print(f"Preprocessed NPZ files found : {len(preproc_files)}")
print(f"Expected completed files     : 3340")
print(f"Directory                    : {PREPROC_DIR}")

if len(preproc_files) == 3340:
    print("STATUS: 3340 preprocessed files detected.")
elif len(preproc_files) > 0:
    print(f"STATUS: {len(preproc_files)} files detected.")
    print("Check the HPC-to-local transfer/path before training.")
else:
    raise FileNotFoundError(
        f"No *_preproc.npz files found under {PREPROC_DIR}"
    )

# Only inspect a small sample here; full QC is Cell 5.
sample = preproc_files[:min(5, len(preproc_files))]
for p in sample:
    with np.load(p) as z:
        data = z["data"]
    print(f"{os.path.basename(p)} -> shape={data.shape}, dtype={data.dtype}")

print("\nNo preprocessing was executed in this cell.")

## CELL 5 — Preprocessing QC
Checks the actual saved tensors before model training: shape, finite values, non-zero brain voxels, and normalization.


In [ ]:

# =========================================================
# CELL 5 — PREPROCESSING QC
# =========================================================
import os
import glob
import numpy as np

preproc_files = sorted(
    glob.glob(os.path.join(PREPROC_DIR, "**", "*_preproc.npz"), recursive=True)
)

print("=" * 60)
print("CELL 5: PREPROCESSING QC")
print("=" * 60)
print(f"Preprocessed files: {len(preproc_files)}")

shape_counts = {}
bad_files = []
voxel_means = []
voxel_stds = []
brain_voxels = []

for path in preproc_files:
    try:
        with np.load(path) as z:
            data = z["data"]

        shape_counts[data.shape] = shape_counts.get(data.shape, 0) + 1

        if not np.isfinite(data).all():
            bad_files.append((path, "NaN/Inf"))
            continue

        mask = data != 0
        n_brain = int(mask.sum())
        brain_voxels.append(n_brain)

        if n_brain == 0:
            bad_files.append((path, "empty brain"))
            continue

        voxel_means.append(float(data[mask].mean()))
        voxel_stds.append(float(data[mask].std()))

    except Exception as e:
        bad_files.append((path, str(e)))

print("Shapes:")
for shape, count in sorted(shape_counts.items(), key=lambda x: str(x[0])):
    print(f"  {shape}: {count}")

if brain_voxels:
    print(f"Brain voxels: min={min(brain_voxels):,}, max={max(brain_voxels):,}")
    print(f"Normalized mean: {np.mean(voxel_means):.4f}")
    print(f"Normalized std : {np.mean(voxel_stds):.4f}")

print(f"Bad files: {len(bad_files)}")
for item in bad_files[:10]:
    print(" ", item)

expected_shape = (160, 192, 160)
valid_shape_count = shape_counts.get(expected_shape, 0)

print(f"\nExpected shape {expected_shape}: {valid_shape_count}/{len(preproc_files)}")

if bad_files:
    print("WARNING: remove/fix bad files before training.")
elif valid_shape_count != len(preproc_files):
    print("WARNING: not every file has the expected 160x192x160 shape.")
else:
    print("QC PASS: all saved tensors have the expected shape and finite values.")
